# Tacit Knowledge Distillation: GLM-5.1 -> Qwen3-8B

Based on Michael Polanyi's insight: **"We know more than we can tell."**

Standard distillation only captures explicit I/O mappings. This pipeline
extracts the teacher's **tacit knowledge** through 3 progressive phases:

| Phase | Polanyi Concept | Method | LR |
|-------|----------------|--------|----|
| 1 | Explicit knowledge | CoT — force reasoning to surface | 1x |
| 2 | Internalization | Direct answers + multi-perspective | 0.5x |
| 3 | Calibrated intuition | Confidence + uncertainty awareness | 0.25x |

**Setup:** Runtime -> Change runtime type -> A100 GPU

## 0. Config

In [ ]:
# ===== FILL THIS =====
ZHIPU_API_KEY = ""  # BigModel API key

# ===== Optional: push to HF Hub =====
HF_TOKEN = ""
HF_REPO_ID = ""  # e.g. "your-name/qwen3-8b-glm5-distill"

# ===== Model =====
STUDENT_MODEL = "Qwen/Qwen3-8B"
TEACHER_MODEL = "glm-5.1"

# ===== Training =====
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 8
BASE_LR = 1e-4
MAX_SEQ_LEN = 2048
LORA_R = 64
LORA_ALPHA = 128

# Phase LR multipliers (Polanyi progression)
PHASE1_LR = 1.0    # Explicit: full speed
PHASE2_LR = 0.5    # Internalization: gentler
PHASE3_LR = 0.25   # Calibration: fine-tune

# ===== Data gen =====
API_WORKERS = 8
TEMPERATURE = 0.7

## 1. Install

In [ ]:
!pip install -q zhipuai transformers datasets peft trl accelerate bitsandbytes flash-attn

import torch, json, time, os, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from zhipuai import ZhipuAI

print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.0f}GB")

## 2. Extract tacit knowledge from GLM-5.1

For each seed prompt, we make **5 API calls** to extract different knowledge dimensions:
- **CoT**: Step-by-step reasoning (tacit → explicit)
- **Direct**: Polished answer (tacit competence)
- **Hot sampling**: High-temperature variant (uncertainty)
- **Perspective**: Alternative angle (subsidiary awareness)
- **Calibrated**: Self-assessed confidence (meta-cognition)

In [ ]:
os.makedirs("data", exist_ok=True)

# System prompts for knowledge extraction
SYS_DIRECT = "You are a helpful, accurate assistant. Provide clear, well-structured responses."

SYS_COT = (
    "You are a helpful assistant that thinks step by step. "
    "Before giving your answer, walk through your reasoning in detail. "
    "Show what you considered, what tradeoffs you weighed, and why you chose your approach. "
    "Format: first your reasoning under '## Thinking', then your answer under '## Answer'."
)

SYS_CONFIDENCE = (
    "You are a calibrated assistant. After answering, add '## Confidence' where you: "
    "1) Rate confidence (high/medium/low) 2) State uncertainties 3) Note what could make your answer wrong."
)

SYS_PERSPECTIVES = [
    "You are an expert who first identifies common misconceptions about the topic, then gives the correct explanation.",
    "You are a Socratic teacher. Guide understanding through a series of progressive questions and answers.",
    "You are a practical engineer. Skip theory — focus on concrete examples, real-world usage, and production gotchas.",
]

SEEDS = [
    {"prompt": "Explain the difference between a stack and a queue with Python examples.", "category": "code"},
    {"prompt": "Write a Python function to find the longest common subsequence of two strings.", "category": "code"},
    {"prompt": "How does garbage collection work in Python? Explain reference counting and the generational collector.", "category": "code"},
    {"prompt": "Implement a simple HTTP server in Python that handles GET and POST requests.", "category": "code"},
    {"prompt": "Write a SQL query to find the second highest salary in each department.", "category": "code"},
    {"prompt": "Implement a binary search tree with insert, delete, and search operations in Python.", "category": "code"},
    {"prompt": "Write a Python decorator that adds retry logic with exponential backoff.", "category": "code"},
    {"prompt": "Implement a simple LRU cache in Python without using functools.", "category": "code"},
    {"prompt": "Write a Python script that processes a CSV file and generates summary statistics.", "category": "code"},
    {"prompt": "Implement the observer design pattern in Python.", "category": "code"},
    {"prompt": "Write a Python function that validates an email address using regex.", "category": "code"},
    {"prompt": "Write a Python asyncio-based web scraper that respects rate limits.", "category": "code"},
    {"prompt": "Implement a thread-safe producer-consumer queue in Python.", "category": "code"},
    {"prompt": "Write a recursive descent parser for simple arithmetic expressions in Python.", "category": "code"},
    {"prompt": "Implement a trie data structure with insert, search, and prefix matching.", "category": "code"},
    {"prompt": "Write a Python context manager for database transactions with rollback on error.", "category": "code"},
    {"prompt": "Implement Dijkstra's shortest path algorithm in Python with a priority queue.", "category": "code"},
    {"prompt": "Write a Python metaclass that automatically adds logging to all methods.", "category": "code"},
    {"prompt": "Implement a simple key-value store with persistence using Python.", "category": "code"},
    {"prompt": "Write a Python generator that reads a large file in chunks without loading it all into memory.", "category": "code"},
    {"prompt": "Explain the CAP theorem in distributed systems with real-world examples.", "category": "reasoning"},
    {"prompt": "What are the trade-offs between microservices and monolithic architecture?", "category": "reasoning"},
    {"prompt": "Explain how transformers work in deep learning, focusing on the attention mechanism.", "category": "reasoning"},
    {"prompt": "What is the difference between TCP and UDP? When would you use each?", "category": "reasoning"},
    {"prompt": "Explain the concept of eventual consistency and how it differs from strong consistency.", "category": "reasoning"},
    {"prompt": "What are the SOLID principles in software engineering? Give examples for each.", "category": "reasoning"},
    {"prompt": "Explain how Docker containers differ from virtual machines.", "category": "reasoning"},
    {"prompt": "What is the difference between process and thread? Explain with examples.", "category": "reasoning"},
    {"prompt": "Explain how HTTPS works, including the TLS handshake process.", "category": "reasoning"},
    {"prompt": "What is the time complexity of Dijkstra's algorithm and why?", "category": "reasoning"},
    {"prompt": "Explain the difference between supervised, unsupervised, and reinforcement learning.", "category": "reasoning"},
    {"prompt": "Explain how B+ trees work and why databases use them for indexing.", "category": "reasoning"},
    {"prompt": "What are the differences between REST, GraphQL, and gRPC? When to use each?", "category": "reasoning"},
    {"prompt": "Explain the concept of backpressure in streaming systems.", "category": "reasoning"},
    {"prompt": "What is the difference between optimistic and pessimistic concurrency control?", "category": "reasoning"},
    {"prompt": "Solve step by step: A train leaves city A at 60 km/h. Another leaves city B (300 km away) at 90 km/h toward A. When and where do they meet?", "category": "math"},
    {"prompt": "Prove that the square root of 2 is irrational.", "category": "math"},
    {"prompt": "Explain the pigeonhole principle and give 3 interesting applications.", "category": "math"},
    {"prompt": "What is the difference between P, NP, and NP-complete problems? Give examples.", "category": "math"},
    {"prompt": "Solve: How many ways can you climb n stairs if you can take 1, 2, or 3 steps at a time? Derive the recurrence.", "category": "math"},
    {"prompt": "请用中文解释什么是知识蒸馏，以及它在大语言模型中的应用。", "category": "chinese"},
    {"prompt": "用Python实现一个简单的聊天机器人框架，支持多轮对话。", "category": "chinese_code"},
    {"prompt": "解释MapReduce的工作原理，并给出一个词频统计的例子。", "category": "chinese"},
    {"prompt": "请解释数据库索引的原理，B+树索引和哈希索引的区别是什么？", "category": "chinese"},
    {"prompt": "用Python实现快速排序算法，并分析其时间和空间复杂度。", "category": "chinese_code"},
    {"prompt": "解释什么是微服务架构，它和单体架构相比有什么优缺点？", "category": "chinese"},
    {"prompt": "用Python实现一个支持过期时间的内存缓存系统。", "category": "chinese_code"},
    {"prompt": "解释分布式系统中的Raft共识算法的核心思想。", "category": "chinese"},
    {"prompt": "请解释Transformer模型中Multi-Head Attention的作用和实现原理。", "category": "chinese"},
    {"prompt": "用Python实现一个简单的正则表达式引擎，支持 . * + 三种操作符。", "category": "chinese_code"},
    {"prompt": "Design a URL shortener system. Describe the API, storage, and how to handle high traffic.", "category": "system_design"},
    {"prompt": "Design a rate limiter. Compare token bucket, leaky bucket, and sliding window approaches.", "category": "system_design"},
    {"prompt": "How would you design a real-time chat application that scales to millions of users?", "category": "system_design"},
    {"prompt": "Explain knowledge distillation in ML. How does it differ from pruning and quantization?", "category": "ml"},
    {"prompt": "What are the key differences between LoRA, QLoRA, and full fine-tuning for LLMs?", "category": "ml"},
    {"prompt": "Explain how RLHF works for aligning language models. What are alternatives like DPO?", "category": "ml"},
    {"prompt": "Write a comprehensive guide to Git branching strategies for a team of 10 developers.", "category": "general"},
    {"prompt": "Explain OAuth 2.0 vs OpenID Connect with practical examples.", "category": "general"},
    {"prompt": "What are the best practices for writing production-ready Python code?", "category": "general"},
    {"prompt": "Explain event-driven architecture and compare it with request-response patterns.", "category": "general"},
]

print(f"{len(SEEDS)} seeds x 5 calls = ~{len(SEEDS)*5} API calls")

In [ ]:
client = ZhipuAI(api_key=ZHIPU_API_KEY)

def call(system, prompt, temp=TEMPERATURE):
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
                temperature=temp, max_tokens=2048,
            )
            return r.choices[0].message.content
        except Exception as e:
            if attempt < 2: time.sleep(2**attempt)
            else: print(f"FAIL: {prompt[:50]}... {e}"); return None

def extract_knowledge(seed):
    """5 API calls per seed: CoT, direct, hot, perspective, calibrated"""
    p = seed["prompt"]
    cat = seed.get("category", "general")
    out = []

    # Phase 1: CoT — tacit reasoning made explicit
    cot = call(SYS_COT, p)
    if cot: out.append({"prompt": p, "response": cot, "category": cat, "phase": 1})

    # Phase 2: Direct — internalized competence
    direct = call(SYS_DIRECT, p)
    if direct: out.append({"prompt": p, "response": direct, "category": cat, "phase": 2})

    # Phase 2: Hot — uncertainty patterns
    hot = call(SYS_DIRECT, p, temp=1.0)
    if hot: out.append({"prompt": p, "response": hot, "category": cat, "phase": 2})

    # Phase 2: Perspective — subsidiary awareness
    persp = call(random.choice(SYS_PERSPECTIVES), p)
    if persp: out.append({"prompt": p, "response": persp, "category": cat, "phase": 2})

    # Phase 3: Calibration — knowing what you don't know
    calib = call(SYS_CONFIDENCE, p)
    if calib: out.append({"prompt": p, "response": calib, "category": cat, "phase": 3})

    return out

# Generate all data
phase_data = {1: [], 2: [], 3: []}

with ThreadPoolExecutor(max_workers=API_WORKERS) as ex:
    futures = {ex.submit(extract_knowledge, s): s for s in SEEDS}
    for f in tqdm(as_completed(futures), total=len(SEEDS), desc="Extracting tacit knowledge"):
        for item in f.result():
            phase_data[item["phase"]].append(item)

# Save
for phase, items in phase_data.items():
    path = f"data/phase{phase}.jsonl"
    with open(path, "w") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Phase {phase}: {len(items)} samples -> {path}")

print(f"\nTotal: {sum(len(v) for v in phase_data.values())} training samples")

In [ ]:
# Quick check
for phase in [1, 2, 3]:
    with open(f"data/phase{phase}.jsonl") as f:
        sample = json.loads(f.readline())
    print(f"\n--- Phase {phase} sample ---")
    print(f"Q: {sample['prompt'][:80]}")
    print(f"A: {sample['response'][:200]}...")

## 3. Progressive Training (Polanyi's 3 Phases)

Same LoRA adapter trained across all 3 phases with decreasing LR.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

def load_phase(phase_num):
    records = []
    path = f"data/phase{phase_num}.jsonl"
    if not os.path.exists(path): return Dataset.from_list([])
    for line in open(path):
        item = json.loads(line.strip())
        records.append({"messages": [
            {"role": "user", "content": item["prompt"]},
            {"role": "assistant", "content": item["response"]},
        ]})
    return Dataset.from_list(records)

# Load model + LoRA
print(f"Loading {STUDENT_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
    attn_implementation="flash_attention_2", trust_remote_code=True,
)

model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))
model.print_trainable_parameters()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
PHASE_NAMES = {1: "Explicit (CoT)", 2: "Internalization", 3: "Calibration"}
PHASE_LR_MULT = {1: PHASE1_LR, 2: PHASE2_LR, 3: PHASE3_LR}

for phase in [1, 2, 3]:
    ds = load_phase(phase)
    if len(ds) == 0:
        print(f"Phase {phase}: no data, skip"); continue

    split = ds.train_test_split(test_size=0.05, seed=42) if len(ds) > 20 else None
    train_ds = split["train"] if split else ds
    eval_ds = split["test"] if split else None
    lr = BASE_LR * PHASE_LR_MULT[phase]

    print(f"\n{'='*50}")
    print(f"Phase {phase}: {PHASE_NAMES[phase]} | {len(train_ds)} samples | LR={lr}")
    print(f"{'='*50}")

    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=f"outputs/phase{phase}",
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=lr,
            lr_scheduler_type="cosine",
            warmup_ratio=0.1,
            weight_decay=0.01,
            bf16=True,
            logging_steps=5,
            save_steps=100,
            save_total_limit=2,
            eval_strategy="steps" if eval_ds else "no",
            eval_steps=100 if eval_ds else None,
            max_seq_length=MAX_SEQ_LEN,
            report_to="none",
            seed=42,
            gradient_checkpointing=True,
        ),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(f"outputs/phase{phase}")
    print(f"Phase {phase} done.")

# Save final adapter
model.save_pretrained("outputs/final-adapter")
tokenizer.save_pretrained("outputs/final-adapter")
print("\nAll 3 phases complete -> outputs/final-adapter")

## 4. Merge & Export

In [ ]:
from peft import PeftModel

del model, trainer
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, "outputs/final-adapter")
model = model.merge_and_unload()

MERGED = "models/merged"
model.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
print(f"Merged -> {MERGED}")

## 5. Test

In [ ]:
del base
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MERGED, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
model.eval()

test_prompts = [
    "Write a Python function to check if a string is a palindrome.",
    "用中文解释什么是梯度下降，以及学习率的作用。",
    "Design a simple task queue system with Redis.",
]

for p in test_prompts:
    msgs = [{"role": "user", "content": p}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True, top_p=0.9)
    dt = time.time() - t0
    gen = out.shape[1] - inputs["input_ids"].shape[1]
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*50}\nQ: {p}\nA: {resp[:400]}\n--- {gen} tok, {dt:.1f}s, {gen/dt:.0f} tok/s ---")

## 6. Push to HF Hub (optional)

In [ ]:
if HF_TOKEN and HF_REPO_ID:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skip hub push. Download models/merged/ manually.")

## 7. Export GGUF for Ollama (optional)

In [ ]:
!pip install -q llama-cpp-python
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp 2>/dev/null; true
!python /tmp/llama.cpp/convert_hf_to_gguf.py {MERGED} --outfile models/model-f16.gguf --outtype f16
!cd /tmp/llama.cpp && make -j quantize 2>/dev/null
!/tmp/llama.cpp/build/bin/llama-quantize models/model-f16.gguf models/model-Q4_K_M.gguf Q4_K_M

size = os.path.getsize("models/model-Q4_K_M.gguf") / 1e9
print(f"\nQ4_K_M: {size:.2f} GB — download this for local Ollama")

In [ ]:
!tar czf /content/distilled-model.tar.gz models/model-Q4_K_M.gguf
from google.colab import files
files.download("/content/distilled-model.tar.gz")